# Prepare data for lion example

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import geopandas as gpd
from shapely import Polygon
import ee, geemap
import geemap.colormaps as cm

In [2]:
from hsa.compute import make_local_dask_client
client = make_local_dask_client(n_workers=10,
                                threads_per_worker=1,
                                local_directory='dask-tmp')
client

/home/molzoo/miniforge3/envs/hsa/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 45655 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:45655/status,
Dashboard: http://127.0.0.1:45655/status,Workers: 10
Total threads: 10,Total memory: 52.38 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:36515,Workers: 0
Dashboard: http://127.0.0.1:45655/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:44655,Total threads: 1
Dashboard: http://127.0.0.1:46207/status,Memory: 5.24 GiB
Nanny: tcp://127.0.0.1:34307,


In [3]:
from hsa.compute import initialize_earth_engine_on_workers
initialize_earth_engine_on_workers(client = client, project = "test-with-greta")

{'tcp://127.0.0.1:34751': True,
 'tcp://127.0.0.1:36301': True,
 'tcp://127.0.0.1:38631': True,
 'tcp://127.0.0.1:39017': True,
 'tcp://127.0.0.1:39067': True,
 'tcp://127.0.0.1:39999': True,
 'tcp://127.0.0.1:42605': True,
 'tcp://127.0.0.1:44655': True,
 'tcp://127.0.0.1:46127': True,
 'tcp://127.0.0.1:46387': True}

In [4]:
# Fetch lion data 
all_lions = gpd.read_file("data/ALL_LION_DATA.csv")

In [5]:
all_lions["Timestamp"] = pd.to_datetime(all_lions["Date"] + " " + all_lions["Time"])

In [8]:
# Subset to NPL
npl_lions = all_lions[all_lions["ID"].str.startswith("NPL")].copy()
sel_npl_lions = npl_lions.groupby("ID").size().sort_values(ascending=False).head(12).index
sel_npl_lions = npl_lions[npl_lions["ID"].isin(sel_npl_lions)].copy()

In [ ]:
# Subset to XPL
xpl_lions = all_lions[all_lions["ID"].str.startswith("XPL")].copy()
sel_xpl_lions = xpl_lions.groupby("ID").size().sort_values(ascending=False).head(12).index
sel_xpl_lions = xpl_lions[xpl_lions["ID"].isin(sel_xpl_lions)].copy()

In [11]:
reloc = npl_lions

In [10]:
reloc = pd.concat([sel_npl_lions, sel_xpl_lions], axis=0)
reloc["Latitude"] = pd.to_numeric(reloc["Latitude"], errors='coerce')
reloc["Longitude"] = pd.to_numeric(reloc["Longitude"], errors='coerce')
reloc = reloc.dropna(subset=["Latitude", "Longitude"])

NameError: name 'sel_xpl_lions' is not defined

In [12]:
from hsa.movement.geometry import prepare_trajectory_data
reloc = prepare_trajectory_data(reloc, id_col = "ID", lat_col = "Latitude", \
    lon_col = "Longitude", source_crs = "EPSG:4326", target_crs = "EPSG:32733")

In [ ]:
reloc.to_csv("data/reloc.csv", index=False)

In [13]:
EE_PROJECT = 'test-with-greta'  # change if needed
TARGET_CRS = 'EPSG:32733'
EXPORT_SCALE = 30
BUFFER_M = 10_000

OUT_ZARR = Path('env_32733_lions.zarr')

START = reloc["Timestamp"].min()
END = reloc["Timestamp"].max()

In [14]:
from hsa.remote_sensing import initialize_earth_engine
e = initialize_earth_engine(project=EE_PROJECT)

perimeter = gpd.GeoDataFrame(geometry=[Polygon.from_bounds(*reloc.total_bounds)], crs=TARGET_CRS)

aoi = gpd.GeoDataFrame(geometry=perimeter.geometry.buffer(BUFFER_M), crs=TARGET_CRS)
aoi_wgs84 = aoi.to_crs('EPSG:4326')
aoi_ee = geemap.geopandas_to_ee(aoi_wgs84)

def reset_map():
    Map = geemap.Map()
    Map.addLayer(aoi_ee, {}, "AOI")
    Map.centerObject(aoi_ee, zoom = 10)
    return Map

In [16]:
Map = reset_map()

In [17]:
sand = ee.Image("ISDASOIL/Africa/v1/sand_content").clip(aoi_ee).select("mean_0_20").rename("sand")
Map.addLayer(sand, {"min": 0, "max": 100}, "sand", False)


In [18]:
Map

Map(center=[-19.521963373400293, 13.774130408623046], controls=(WidgetControl(options=['position', 'transparen…

In [ ]:
def mask_s2_clouds(image):
    qa = image.select("QA60")

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )

    optical = (
        image.updateMask(mask)
        .select(
            ["B2", "B3", "B4", "B8", "B11", "B12"],
            ["blue", "green", "red", "nir", "swir1", "swir2"],
        )
        .divide(10000)
    )

    return optical.copyProperties(image, ["system:time_start"])

In [ ]:
def add_water_index(image):
    mndwi = image.normalizedDifference(["green", "swir1"]).rename("mndwi")
    return image.addBands(mndwi)

In [ ]:
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi_ee.geometry())
    .filterDate(START, END)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
    .map(mask_s2_clouds)
    .map(add_water_index)
)

In [ ]:
mndwi_water = s2.select("mndwi").max().clip(aoi_ee)

Map = reset_map()
Map.addLayer(mndwi_water, {"min": 0, "max": 1, "palette": cm.palettes.Blues}, "MNDWI")
Map.addLayer(mndwi_water.gt(0.6))
Map

In [ ]:
water = mndwi_water.gt(0.6)
land_mask = water.Not()

In [ ]:
# Sentinel-1
s1 = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.eq("orbitProperties_pass", "ASCENDING"))
    .filterDate(START, END)
    .filterBounds(aoi_ee.geometry())
)

In [ ]:
# ---------------------
# Calculate S1 Indices
# ---------------------
def add_s1_indices(image):

    vv = image.select("VV")
    vh = image.select("VH")

    vv_vh_diff = vv.subtract(vh).rename("vv_vh_diff")

    vv_lin = ee.Image(10).pow(vv.divide(10)).rename("vv_linear")
    vh_lin = ee.Image(10).pow(vh.divide(10)).rename("vh_linear")

    vv_vh_ratio = vv_lin.divide(vh_lin).rename("vv_vh_ratio")

    return image.addBands([
        vv_vh_diff,
        vv_lin,
        vh_lin,
        vv_vh_ratio,
    ])

s1 = s1.map(add_s1_indices)

In [ ]:
def add_s2_indices(image):
    ndvi = image.normalizedDifference(["nir", "red"]).rename("ndvi")
    ndmi = image.normalizedDifference(["nir", "swir1"]).rename("ndmi")
    mndwi = image.normalizedDifference(["green", "swir1"]).rename("mndwi")

    bsi = (
        image.select("swir1").add(image.select("red"))
        .subtract(image.select("nir")).subtract(image.select("blue"))
        .divide(
            image.select("swir1").add(image.select("red"))
            .add(image.select("nir")).add(image.select("blue"))
        )
        .rename("bsi")
    )

    savi = (
        image.select("nir").subtract(image.select("red"))
        .multiply(1.5)
        .divide(image.select("nir").add(image.select("red")).add(0.5))
        .rename("savi")
    )

    return image.addBands([ndvi, ndmi, mndwi, bsi, savi])

s2 = s2.map(add_s2_indices)

In [ ]:
Map = reset_map()
Map

In [ ]:
vv_median = s1.select("VV").median().clip(aoi_ee)
vh_median = s1.select("VH").median().clip(aoi_ee)
vv_vh_diff_median = s1.select("vv_vh_diff").median().clip(aoi_ee)

Map.addLayer(
    ee.Image.cat([vv_median, vh_median, vv_vh_diff_median]),
    {
        "bands": ["VV", "VH", "vv_vh_diff"],
        "min": [-18, -25, 2],
        "max": [-5, -12, 12],
    },
    "S1 False Color Composite")

red = s2.select("red").median().clip(aoi_ee)
green = s2.select("green").median().clip(aoi_ee)
blue = s2.select("blue").median().clip(aoi_ee)

Map.addLayer(
    ee.Image.cat([red, green, blue]),
    {
        "bands": ["red", "green", "blue"],
        "min": 0.02,
        "max": 0.35,
    },
    "S2 True Colour Composite"
)

In [ ]:
Map.addLayer(s1.select("vv_vh_ratio").median().clip(aoi_ee), {"min": 1, "max": 8, "palette": cm.palettes.inferno}, "VV/VH")
Map.addLayer(s2.select("ndvi").median().clip(aoi_ee), {"min": 0, "max": 0.55, "palette": cm.palettes.Greens}, "NDVI")
Map.addLayer(s2.select("savi").median().clip(aoi_ee), {"min": 0, "max": 0.55, "palette": cm.palettes.Greens}, "SAVI")
Map.addLayer(s2.select("bsi").median().clip(aoi_ee), {"min": -0.5, "max": 0.5, "palette": cm.palettes.hot}, "BSI")

In [ ]:
flow_acc = (
    ee.Image("WWF/HydroSHEDS/15ACC")
    .select("b1")
    .clip(aoi_ee)
    .rename("flow_acc")
)

stream_threshold = 1000

streams = (
    flow_acc
    .gte(stream_threshold)
    .unmask(0)
    .rename("stream")
)

stream_projection = streams.projection()

print(stream_projection.getInfo())
print("Nominal scale:", stream_projection.nominalScale().getInfo())

max_distance_m = 100_000
pixel_size_m = stream_projection.nominalScale()

max_distance_pixels = (
    ee.Number(max_distance_m)
    .divide(pixel_size_m)
    .ceil()
)

distance_squared_pixels = streams.fastDistanceTransform(
    neighborhood=max_distance_pixels,
    units="pixels",
    metric="squared_euclidean",
)

distance_to_river = (
    distance_squared_pixels
    .sqrt()
    .multiply(pixel_size_m)
    .rename("distance_to_river_m")
    .clip(aoi_ee)
)

Map.addLayer(
    streams.selfMask(),
    {"palette": ["00ffff"]},
    "HydroSHEDS streams",
)

Map.addLayer(
    distance_to_river,
    {
        "min": 0,
        "max": 100_000,
        "palette": [
            "08306b",
            "2171b5",
            "6baed6",
            "c6dbef",
            "ffffcc",
        ],
    },
    "Distance to river",
)

In [ ]:
dem = ee.Image("USGS/SRTMGL1_003").clip(aoi_ee).select("elevation")
slope = ee.Terrain.slope(dem).rename("slope")

In [ ]:
stack = ee.Image.cat([
    dem,
    slope,
    s2.select("ndvi").median().clip(aoi_ee),
    s1.select("vv_vh_ratio").median().clip(aoi_ee),
    distance_to_river
])

In [ ]:
from hsa.remote_sensing import ee_image_to_xarray_stack
from hsa.compute import suggest_xy_chunks, persist_if_dask, write_raster_stack_zarr

env = ee_image_to_xarray_stack(
    stack,
    geometry=aoi_ee.geometry(),
    crs="EPSG:32733",
    scale=100,
)

In [ ]:
env = write_raster_stack_zarr(env, "lion_predictor_stack.zarr")